# MyAgent — OneClick Deployment on AMD Radeon Cloud

**Track 2: Private AI Agent Development & Local Deployment**

This notebook automates the full deployment of MyAgent on an AMD ROCm instance (W7900 / 48GB).

Run each cell in order. Total time: ~30-40 min (model download is the bottleneck).

---
## Step 0 — Environment Check

Verify we're running on the expected AMD GPU.

In [ ]:
import subprocess, json, os, sys, time

print("=== GPU ===")
r = subprocess.run(["rocminfo"], capture_output=True, text=True)
for line in r.stdout.split("\n"):
    if "Marketing Name" in line:
        print(line.strip())

print("\n=== VRAM ===")
r = subprocess.run(["rocm-smi", "--showmeminfo", "vram", "--csv"], capture_output=True, text=True)
for line in r.stdout.strip().split("\n")[1:]:
    parts = line.split(",")
    if len(parts) >= 2 and parts[1].strip().isdigit():
        gb = int(parts[1].strip()) / (1024**3)
        print(f"  {gb:.1f} GB")

print("\n=== ROCm ===")
r = subprocess.run(["cat", "/opt/rocm/.info/version"], capture_output=True, text=True)
print(r.stdout.strip() if r.stdout.strip() else "unknown")

print("\n=== Python ===")
print(sys.version)

---
## Step 1 — Install Python Dependencies

FastAPI, WebSocket, starlette, and other backend requirements.

In [ ]:
!pip install -q fastapi uvicorn[standard] websockets python-multipart pyyaml httpx Pillow

# Verify key packages
import importlib
for pkg in ["fastapi", "uvicorn", "websockets", "yaml", "PIL"]:
    try:
        m = importlib.import_module(pkg if pkg != "PIL" else "PIL")
        print(f"  ✅ {pkg}")
    except ImportError:
        print(f"  ❌ {pkg} — FAILED")

---
## Step 2 — Download GGUF Model

Downloads Qwen2.5-7B-Instruct-GGUF (Q4_K_M, ~4.9 GB). Tries ModelScope first (fastest in China), then HuggingFace.

In [ ]:
import os, subprocess, urllib.request, hashlib, shutil

MODEL_DIR = os.path.expanduser("~/models")
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_FILE = os.path.join(MODEL_DIR, "Qwen2.5-7B-Instruct-Q4_K_M.gguf")
EXPECTED_SHA256 = "e179fc26d5c4f70b01e08f67c6ca8f7c70688ee5af983f2a00f3941fdcd0d7c6"

if os.path.exists(MODEL_FILE):
    print(f"Model already exists: {MODEL_FILE}")
    print(f"Size: {os.path.getsize(MODEL_FILE) / (1024**3):.2f} GB")
else:
    # Try ModelScope (China-friendly, fastest)
    url_ms = "https://www.modelscope.cn/models/aiclub/Qwen2.5-7B-Instruct-GGUF/resolve/master/Qwen2.5-7B-Instruct-Q4_K_M.gguf"
    url_hf = "https://huggingface.co/Qwen/Qwen2.5-7B-Instruct-GGUF/resolve/main/Qwen2.5-7B-Instruct-Q4_K_M.gguf"
    
    for i, url in enumerate([url_ms, url_hf]):
        source = "ModelScope" if i == 0 else "HuggingFace"
        print(f"Trying {source}: {url[:80]}...")
        try:
            # Use curl for progress display & resume support
            cmd = ["curl", "-L", "-o", MODEL_FILE, "--retry", "3", "-C", "-"]
            if i == 0:
                cmd.extend(["--connect-timeout", "15"])
            r = subprocess.run(cmd, timeout=600, capture_output=True, text=True)
            if os.path.exists(MODEL_FILE) and os.path.getsize(MODEL_FILE) > 100_000_000:
                print(f"✅ Downloaded from {source}")
                break
            else:
                print(f"❌ Failed from {source}, trying next...")
                if os.path.exists(MODEL_FILE):
                    os.remove(MODEL_FILE)
        except Exception as e:
            print(f"❌ Error from {source}: {e}")
    
    if not os.path.exists(MODEL_FILE):
        raise RuntimeError("All download sources failed!")

# Verify integrity
print("\nVerifying SHA256...")
sha = hashlib.sha256()
with open(MODEL_FILE, "rb") as f:
    for chunk in iter(lambda: f.read(8192*1024), b""):
        sha.update(chunk)
got = sha.hexdigest()
if got == EXPECTED_SHA256:
    print(f"✅ SHA256 OK: {got[:16]}...")
else:
    print(f"⚠️ SHA256 mismatch! got={got[:16]}... expected={EXPECTED_SHA256[:16]}...")
    print("Proceeding anyway — model may still work.")

print(f"\nFinal size: {os.path.getsize(MODEL_FILE) / (1024**3):.2f} GB")

---
## Step 3 — Get llama.cpp (ROCm prebuilt binary)

Downloads the **official prebuilt ROCm 7.2 binary** (~124 MB, <1 min).

No source compilation needed — the official release matches this container's ROCm 7.2.1 runtime.
Falls back to source build only if the download fails.

In [ ]:
import os, subprocess, tarfile, glob, shutil, urllib.request

LLAMA_CPP_DIR = os.path.expanduser("~/llama.cpp")
BUILD_DIR = os.path.join(LLAMA_CPP_DIR, "build")
SERVER_BIN = os.path.join(BUILD_DIR, "bin", "llama-server")

TAG = "b10267"
URL = f"https://github.com/ggml-org/llama.cpp/releases/download/{TAG}/llama-{TAG}-bin-ubuntu-rocm-7.2-x64.tar.gz"

if os.path.exists(SERVER_BIN):
    print(f"✅ Already present: {SERVER_BIN}")
else:
    os.makedirs(BUILD_DIR, exist_ok=True)
    tmp_tar = "/tmp/llama-rocm.tar.gz"
    ok = False

    print(f"Downloading official ROCm 7.2 prebuilt ({TAG}, ~124 MB)…")
    try:
        subprocess.run(["curl", "-fL", "--retry", "3", "--connect-timeout", "20",
                        "-o", tmp_tar, URL], check=True)
        size = os.path.getsize(tmp_tar)
        print(f"  downloaded: {size/1e6:.1f} MB")
        if size > 50_000_000:
            with tarfile.open(tmp_tar) as tf:
                tf.extractall(BUILD_DIR)
            # Official archive may nest under build/bin or bin — normalize either way
            if not os.path.exists(SERVER_BIN):
                found = glob.glob(os.path.join(BUILD_DIR, "**", "llama-server"), recursive=True)
                if found:
                    src_dir = os.path.dirname(found[0])
                    os.makedirs(os.path.dirname(SERVER_BIN), exist_ok=True)
                    for f in os.listdir(src_dir):
                        shutil.copy2(os.path.join(src_dir, f),
                                     os.path.join(os.path.dirname(SERVER_BIN), f))
            for f in glob.glob(os.path.join(BUILD_DIR, "bin", "*")):
                os.chmod(f, 0o755)
            ok = os.path.exists(SERVER_BIN)
        os.remove(tmp_tar)
    except Exception as e:
        print(f"  ⚠️ prebuilt download failed: {e}")

    if ok:
        print("✅ Prebuilt ready — skipped ~15 min of compilation.")
    else:
        print("⚠️ Falling back to source build (10-15 min, screen may look frozen — normal)…")
        if not os.path.exists(os.path.join(LLAMA_CPP_DIR, ".git")):
            shutil.rmtree(LLAMA_CPP_DIR, ignore_errors=True)
            subprocess.run(["git", "clone", "--depth", "1",
                            "https://github.com/ggml-org/llama.cpp.git",
                            LLAMA_CPP_DIR], check=True)
        os.makedirs(BUILD_DIR, exist_ok=True)
        subprocess.run(["cmake", "..", "-DGGML_HIP=ON",
                        "-DAMDGPU_TARGETS=gfx1100",
                        "-DCMAKE_C_COMPILER=hipcc",
                        "-DCMAKE_CXX_COMPILER=hipcc",
                        "-DCMAKE_BUILD_TYPE=Release"], cwd=BUILD_DIR, check=True)
        subprocess.run(["cmake", "--build", ".", "--config", "Release",
                        "-j", str(os.cpu_count() or 8)], cwd=BUILD_DIR, check=True)

assert os.path.exists(SERVER_BIN), f"llama-server not found at {SERVER_BIN}"
print(f"\n✅ Ready: {SERVER_BIN}")
subprocess.run([SERVER_BIN, "--version"], timeout=30)

---
## Step 4 — Configure MyAgent

Set up paths and single-GPU mode in settings.

In [ ]:
import yaml, os

# Detect notebook directory (repo root)
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.exists(os.path.join(REPO_ROOT, "backend")):
    REPO_ROOT = os.getcwd()  # fallback

SETTINGS_PATH = os.path.join(REPO_ROOT, "backend", "core", "config", "settings.py")

print(f"Repo root: {REPO_ROOT}")
print(f"Settings: {SETTINGS_PATH}")
print(f"Exists: {os.path.exists(SETTINGS_PATH)}")

# Write runtime config overlay
config_overlay = {
    "model_path": os.path.expanduser("~/models/Qwen2.5-7B-Instruct-Q4_K_M.gguf"),
    "host": "0.0.0.0",
    "port": 8000,
    "n_gpu_layers": -1,  # all layers to GPU
    "ctx_size": 8192,
    "single_gpu_mode": True,
}

overlay_path = os.path.join(REPO_ROOT, "runtime_config.json")
import json
with open(overlay_path, "w") as f:
    json.dump(config_overlay, f, indent=2)
print(f"\n✅ Runtime config written to {overlay_path}")
print(json.dumps(config_overlay, indent=2))

---
## Step 5 — Start LLM Inference Engine (llama-server)

Starts the llama.cpp server in the background. This loads the model into GPU VRAM (~5 GB).

In [ ]:
import subprocess, os, time, signal

LLAMA_SERVER = os.path.expanduser("~/llama.cpp/build/bin/llama-server")
MODEL_PATH = os.path.expanduser("~/models/Qwen2.5-7B-Instruct-Q4_K_M.gguf")

# Kill any existing llama-server
subprocess.run(["pkill", "-f", "llama-server"], capture_output=True)
time.sleep(2)

print("Starting llama-server (loading model into GPU)…")
proc = subprocess.Popen([
    LLAMA_SERVER,
    "-m", MODEL_PATH,
    "--host", "0.0.0.0",
    "--port", "8000",
    "-ngl", "99",          # offload all layers to GPU
    "-c", "8192",         # context window
    "--parallel", "4",
    "-t", str(os.cpu_count() or 8),
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

# Wait for server to be ready (up to 120s for model loading)
print("Waiting for server ready (model loading takes 30-60s)…")
start = time.time()
while time.time() - start < 120:
    try:
        import urllib.request
        req = urllib.request.urlopen("http://localhost:8000/v1/models", timeout=3)
        print(f"\n✅ Server ready in {time.time()-start:.1f}s!")
        print(req.read().decode()[:200])
        break
    except:
        time.sleep(3)
        elapsed = int(time.time() - start)
        if elapsed % 10 == 0:
            print(f"  ... {elapsed}s elapsed, still loading…")
else:
    print("\n⚠️ Timeout waiting for server. Check logs below:")
    out = proc.stdout.read().decode(errors="replace")[-2000:]
    print(out)

---
## Step 6 — Start Backend API (FastAPI + WebSocket)

Starts the MyAgent backend on port 8080.

In [ ]:
import subprocess, time, sys, os

REPO_ROOT = os.getcwd() if os.path.exists("backend") else os.path.abspath(os.path.join(os.getcwd(), ".."))
BACKEND_DIR = os.path.join(REPO_ROOT, "backend")

# Kill existing backend
subprocess.run(["pkill", "-f", "uvicorn.*main"], capture_output=True)
time.sleep(1)

# Add backend to path so imports work
sys.path.insert(0, BACKEND_DIR)
os.chdir(BACKEND_DIR)

# Install backend deps if needed
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               capture_output=True)

print("Starting FastAPI backend on port 8080…")
backend_proc = subprocess.Popen([
    sys.executable, "-m", "uvicorn", "main:app",
    "--host", "0.0.0.0",
    "--port", "8080",
    "--reload",
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, cwd=BACKEND_DIR)

# Wait for backend
time.sleep(8)
try:
    import urllib.request
    req = urllib.request.urlopen("http://localhost:8080/api/health", timeout=5)
    print(f"✅ Backend healthy: {req.read().decode()[:100]}")
except Exception as e:
    print(f"⚠️ Backend health check: {e}")
    out = backend_proc.stdout.read().decode(errors="replace")[-1500:]
    print(out)

---
## Step 7 — Access URLs

Your instance is now running! Open these URLs:

In [ ]:
import urllib.request

# Get public IP
try:
    public_ip = urllib.request.urlopen("https://api.ipify.org?format=text", timeout=5).read().decode()
except:
    public_ip = urllib.request.urlopen("https://ifconfig.me", timeout=5).read().decode()

print("=" * 60)
print("  🎉 MYAGENT IS RUNNING!")
print("=" * 60)
print(f"\n  Frontend (Vue3 SPA):     http://{public_ip}/")
print(f"  Backend API docs:         http://{public_ip}:8080/docs")
print(f"  LLM Inference Engine:     http://{public_ip}:8000/v1/models")
print(f"\n  ⚠️ If frontend shows 'connection refused', open port 80 in security group.")
print("=" * 60)

---
## Step 8 — Quick Smoke Test

Send a test message through the backend to verify the full pipeline works.

In [ ]:
import json, urllib.request, time

# Test via direct llama.cpp API
payload = json.dumps({
    "model": "Qwen2.5-7B-Instruct-Q4_K_M.gguf",
    "messages": [{"role": "user", "content": "Say hello in one sentence."}],
    "max_tokens": 50,
    "temperature": 0.7,
}).encode()

req = urllib.request.Request(
    "http://localhost:8000/v1/chat/completions",
    data=payload,
    headers={"Content-Type": "application/json"},
)

print("Sending test message to LLM…")
start = time.time()
resp = urllib.request.urlopen(req, timeout=60)
elapsed = time.time() - start
result = json.loads(resp.read())

reply = result["choices"][0]["message"]["content"]
usage = result.get("usage", {})

print(f"\n✅ Response ({elapsed:.1f}s):")
print(f"  {reply}")
print(f"\nTokens: prompt={usage.get('prompt_tokens','?')}, completion={usage.get('completion_tokens','?')}")
if usage.get('prompt_tokens') and usage.get('completion_tokens') and elapsed > 0:
    tps = usage['completion_tokens'] / elapsed
    print(f"Speed: ~{tps:.1f} tokens/sec")
    print(f"TTFT: estimated ~{elapsed - (usage['completion_tokens']/tps if tps > 0 else 0):.1f}s")